# 01 — Setup & EDA
**Hybrid CNN Fusion for Pneumonia Detection**

This notebook covers:
1. Download the Kaggle chest X-ray dataset
2. Merge + stratified re-split (80/10/10)
3. Build preprocessing pipeline
4. Exploratory data analysis
5. Pipeline sanity check
6. MLflow initialisation

---

## 0. Environment

In [ ]:
import os
os.chdir('/content/MINI_PROJECT')
print(f'Working directory: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
from pathlib import Path

import torch
from torchvision import transforms

from src.utils import set_seed, get_device, load_config
from src.dataset import (
    collect_image_paths,
    stratified_split,
    get_transforms,
    get_dataloaders,
    compute_class_weights,
)

set_seed(42)
device = get_device()
config = load_config()

print('All imports ready')

## 1. Download dataset

Choose **one** of the options below.

In [ ]:
# ── OPTION A: Kaggle API ──
# Upload your kaggle.json first: Colab sidebar → Files → Upload

# !pip install -q kaggle
# !mkdir -p ~/.kaggle
# !cp /content/kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data/
# !unzip -q data/chest-xray-pneumonia.zip -d data/

In [ ]:
# ── OPTION B: Direct download ──
# If Kaggle API doesn't work, download manually from:
# https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
# Upload the zip to Google Drive, then:

# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/chest-xray-pneumonia.zip data/
# !unzip -q data/chest-xray-pneumonia.zip -d data/

In [ ]:
# Verify dataset is in place
raw_dir = config['data']['raw_dir']
!find {raw_dir} -type f -name '*.jpeg' | wc -l
!ls {raw_dir}

## 2. Collect image paths + stratified re-split

In [ ]:
# Collect all paths
df = collect_image_paths(config['data']['raw_dir'])

In [ ]:
# Show the problem: original val split is tiny
print('Original split sizes:')
print(df.groupby(['original_split', 'label']).size().unstack(fill_value=0))
print(f'\nVal set only has {len(df[df.original_split == "val"])} images — not usable!')

In [ ]:
# Stratified re-split
train_df, val_df, test_df = stratified_split(
    df,
    split_ratios=config['data']['split_ratios'],
    seed=config['data']['random_seed'],
    save_path='data/split_indices.csv',
)

## 3. EDA — Class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

label_names = {0: 'Normal', 1: 'Pneumonia'}
colors = ['#5DCAA5', '#D85A30']

for ax, (name, split_df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    counts = split_df['label'].value_counts().sort_index()
    bars = ax.bar(
        [label_names[i] for i in counts.index],
        counts.values,
        color=colors,
        edgecolor='white',
        linewidth=0.5,
    )
    ax.set_title(f'{name} (n={len(split_df)})', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')

    # Add count labels on bars
    for bar, count in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            str(count), ha='center', fontsize=11,
        )

plt.suptitle('Class Distribution Across Splits', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Print imbalance ratio
train_counts = Counter(train_df['label'].values)
ratio = train_counts[1] / train_counts[0]
print(f'\nImbalance ratio (Pneumonia:Normal) = {ratio:.2f}:1')

## 4. EDA — Sample images

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for row, (label, label_name) in enumerate([(0, 'Normal'), (1, 'Pneumonia')]):
    samples = train_df[train_df['label'] == label].sample(6, random_state=42)
    for col, (_, sample) in enumerate(samples.iterrows()):
        img = Image.open(sample['filepath']).convert('L')  # grayscale for display
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(label_name, fontsize=10)
        axes[row, col].axis('off')

plt.suptitle('Sample Chest X-rays', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. EDA — Image statistics

In [ ]:
# Sample 200 images to check sizes and pixel stats (faster than loading all)
sample_df = train_df.sample(min(200, len(train_df)), random_state=42)

widths, heights, mean_pixels = [], [], []

for _, row in sample_df.iterrows():
    img = Image.open(row['filepath'])
    w, h = img.size
    widths.append(w)
    heights.append(h)
    mean_pixels.append(np.array(img).mean())

print('Image size statistics (from 200 samples):')
print(f'  Width  — min: {min(widths)}, max: {max(widths)}, median: {int(np.median(widths))}')
print(f'  Height — min: {min(heights)}, max: {max(heights)}, median: {int(np.median(heights))}')
print(f'  Mode   — {Image.open(sample_df.iloc[0]["filepath"]).mode}')
print(f'\nPixel intensity (0-255):')
print(f'  Mean: {np.mean(mean_pixels):.1f}, Std: {np.std(mean_pixels):.1f}')

In [ ]:
# Resolution scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(widths, heights, alpha=0.4, s=15, c='#534AB7')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Height (px)')
axes[0].set_title('Original image resolutions')
axes[0].axhline(y=224, color='red', linestyle='--', alpha=0.5, label='Target 224')
axes[0].axvline(x=224, color='red', linestyle='--', alpha=0.5)
axes[0].legend()

# Pixel intensity by class
normal_pixels = [np.array(Image.open(row['filepath'])).mean()
                 for _, row in sample_df[sample_df['label']==0].head(80).iterrows()]
pneumonia_pixels = [np.array(Image.open(row['filepath'])).mean()
                    for _, row in sample_df[sample_df['label']==1].head(80).iterrows()]

axes[1].hist(normal_pixels, bins=25, alpha=0.6, color='#5DCAA5', label='Normal')
axes[1].hist(pneumonia_pixels, bins=25, alpha=0.6, color='#D85A30', label='Pneumonia')
axes[1].set_xlabel('Mean pixel intensity')
axes[1].set_ylabel('Count')
axes[1].set_title('Pixel intensity distribution by class')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/figures/image_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Build DataLoaders + sanity check

In [ ]:
# Build everything in one call
loaders = get_dataloaders(config)

In [ ]:
# Sanity check: grab one batch
images, labels = next(iter(loaders['train']))

print(f'Batch shape:  {images.shape}')      # should be [32, 3, 224, 224]
print(f'Label shape:  {labels.shape}')       # should be [32]
print(f'Pixel range:  [{images.min():.2f}, {images.max():.2f}]')
print(f'Label values: {sorted(labels.unique().tolist())}')
print(f'Batch labels: {dict(Counter(labels.numpy()))}')

In [ ]:
# Visualise augmented samples from the batch
# (need to un-normalise for display)
mean = torch.tensor(config['preprocessing']['imagenet_mean']).view(3, 1, 1)
std = torch.tensor(config['preprocessing']['imagenet_std']).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
label_names = {0: 'Normal', 1: 'Pneumonia'}

for i, ax in enumerate(axes.flat):
    img = images[i] * std + mean        # un-normalise
    img = img.clamp(0, 1)               # clip to valid range
    img = img.permute(1, 2, 0).numpy()  # CHW → HWC
    ax.imshow(img)
    ax.set_title(f'{label_names[labels[i].item()]}', fontsize=10)
    ax.axis('off')

plt.suptitle('Augmented training batch (after transforms)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPipeline sanity check PASSED')

## 7. MLflow init

In [ ]:
import mlflow

mlflow.set_tracking_uri(config['paths']['mlflow_uri'])
mlflow.set_experiment(config['experiment']['name'])

# Test run
with mlflow.start_run(run_name='setup-test'):
    mlflow.log_param('dataset_size', len(loaders['train'].dataset) + len(loaders['val'].dataset) + len(loaders['test'].dataset))
    mlflow.log_param('batch_size', config['training']['batch_size'])
    mlflow.log_param('image_size', config['preprocessing']['image_size'])
    mlflow.log_metric('dummy_accuracy', 0.0)
    print('MLflow test run logged successfully')

## Summary

**What we've confirmed:**
- Dataset downloaded and verified
- Stratified 80/10/10 split (saved to `data/split_indices.csv`)
- Preprocessing pipeline works: resize → augment → normalise
- DataLoaders produce correct tensor shapes
- Class weights computed for imbalanced loss
- MLflow tracking initialised

**Next:** Open `notebooks/02_baselines.ipynb` to start training.

---
### Save to GitHub

In [ ]:
# Run before closing Colab!
!git add .
!git commit -m "setup: complete EDA notebook with data pipeline"
!git push origin main